In [ ]:
# Loading the dataset using pandas

import pandas as pd

df = pd.read_csv('customer_shopping_behavior.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Summary statistics using .describe()
df.describe(include='all')

In [ ]:
# Checking if missing data or null values are present in the dataset

df.isnull().sum()

In [ ]:
# Imputing missing values in Review Rating column with the median rating of the product category

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [ ]:
# Renaming columns according to snake casing for better readability and documentation

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [ ]:
df.columns

In [ ]:
# create a new column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [ ]:
df[['age','age_group']].head(10)

In [ ]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

In [ ]:
df[['discount_applied','promo_code_used']].head(10)

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
# Dropping promo code used column

df = df.drop('promo_code_used', axis=1)

In [ ]:
df.columns

# Connecting to SQLlite

In [ ]:
import sqlite3

conn = sqlite3.connect("customer_behavior.db")

In [ ]:
df.to_sql(
    "customer",
    conn,
    if_exists="replace",
    index=False
)

In [ ]:
query = """
SELECT *
FROM customer
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
query = """
SELECT COUNT(*) AS total_records
FROM customer;
"""

pd.read_sql_query(query, conn)

In [ ]:
query = """
PRAGMA table_info(customer);
"""

pd.read_sql_query(query, conn)

In [ ]:
query = """
SELECT
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_missing,
    SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS age_missing,
    SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) AS gender_missing,
    SUM(CASE WHEN item_purchased IS NULL THEN 1 ELSE 0 END) AS item_purchased_missing,
    SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS category_missing,
    SUM(CASE WHEN purchase_amount IS NULL THEN 1 ELSE 0 END) AS purchase_amount_missing
FROM customer;
"""

pd.read_sql_query(query, conn)

In [ ]:
query = """
SELECT customer_id, COUNT(*) AS count
FROM customer
GROUP BY customer_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)

In [ ]:
query = """
SELECT
    category,
    COUNT(*) AS total_purchases
FROM customer
GROUP BY category
ORDER BY total_purchases DESC;
"""

category_purchases = pd.read_sql_query(query, conn)

category_purchases

In [ ]:
query = """
SELECT
    category,
    SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY category
ORDER BY total_revenue DESC;
"""

category_revenue = pd.read_sql_query(query, conn)

category_revenue

In [ ]:
query = """
SELECT
    category,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY category
ORDER BY average_purchase DESC;
"""

avg_category_purchase = pd.read_sql_query(query, conn)

avg_category_purchase

In [ ]:
query = """
SELECT
    gender,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY gender
ORDER BY total_revenue DESC;
"""

gender_analysis = pd.read_sql_query(query, conn)

gender_analysis

In [ ]:
query = """
SELECT
    subscription_status,
    COUNT(*) AS customers,
    ROUND(AVG(purchase_amount), 2) AS average_purchase,
    SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY subscription_status
ORDER BY total_revenue DESC;
"""

subscription_analysis = pd.read_sql_query(query, conn)

subscription_analysis

In [ ]:
query = """
SELECT
    season,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY season
ORDER BY total_revenue DESC;
"""

season_analysis = pd.read_sql_query(query, conn)

season_analysis

In [ ]:
query = """
SELECT
    location,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY location
ORDER BY total_revenue DESC;
"""

location_analysis = pd.read_sql_query(query, conn)

location_analysis.head(10)

In [ ]:
query = """
SELECT
    customer_id,
    age,
    gender,
    location,
    category,
    item_purchased,
    purchase_amount
FROM customer
ORDER BY purchase_amount DESC
LIMIT 10;
"""

top_customers = pd.read_sql_query(query, conn)

top_customers

In [ ]:
query = """
SELECT
    item_purchased,
    COUNT(*) AS total_purchases
FROM customer
GROUP BY item_purchased
ORDER BY total_purchases DESC;
"""

item_purchases = pd.read_sql_query(query, conn)

item_purchases

In [ ]:
query = """
SELECT
    item_purchased,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY item_purchased
ORDER BY total_revenue DESC;
"""

item_revenue = pd.read_sql_query(query, conn)

item_revenue

In [ ]:
query = """
SELECT
    ROUND(AVG(review_rating), 2) AS average_rating,
    MIN(review_rating) AS minimum_rating,
    MAX(review_rating) AS maximum_rating
FROM customer;
"""

rating_summary = pd.read_sql_query(query, conn)

rating_summary

In [ ]:
query = """
SELECT
    category,
    ROUND(AVG(review_rating), 2) AS average_rating,
    COUNT(*) AS total_reviews
FROM customer
GROUP BY category
ORDER BY average_rating DESC;
"""

rating_category = pd.read_sql_query(query, conn)

rating_category

In [ ]:
query = """
SELECT
    discount_applied,
    COUNT(*) AS total_purchases,
    ROUND(AVG(purchase_amount), 2) AS average_purchase,
    SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY discount_applied
ORDER BY total_revenue DESC;
"""

discount_analysis = pd.read_sql_query(query, conn)

discount_analysis

In [ ]:
query = """
SELECT
    payment_method,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY payment_method
ORDER BY total_purchases DESC;
"""

payment_analysis = pd.read_sql_query(query, conn)

payment_analysis

In [ ]:
query = """
SELECT
    shipping_type,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY shipping_type
ORDER BY total_purchases DESC;
"""

shipping_analysis = pd.read_sql_query(query, conn)

shipping_analysis

In [ ]:
query = """
SELECT
    frequency_of_purchases,
    COUNT(*) AS total_customers,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY frequency_of_purchases
ORDER BY total_customers DESC;
"""

frequency_analysis = pd.read_sql_query(query, conn)
frequency_analysis

In [ ]:
query = """
SELECT
    CASE
        WHEN age < 25 THEN '18-24'
        WHEN age BETWEEN 25 AND 34 THEN '25-34'
        WHEN age BETWEEN 35 AND 44 THEN '35-44'
        WHEN age BETWEEN 45 AND 54 THEN '45-54'
        ELSE '55+'
    END AS calculated_age_group,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY calculated_age_group
ORDER BY total_revenue DESC;
"""

age_analysis = pd.read_sql_query(query, conn)
age_analysis

In [ ]:
query = """
SELECT
    previous_purchases,
    COUNT(*) AS total_customers,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY previous_purchases
ORDER BY previous_purchases;
"""

previous_purchase_analysis = pd.read_sql_query(query, conn)
previous_purchase_analysis

In [ ]:
query = """
SELECT
    frequency_of_purchases,
    COUNT(*) AS total_customers,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY frequency_of_purchases
ORDER BY total_revenue DESC;
"""

frequency_revenue_analysis = pd.read_sql_query(query, conn)
frequency_revenue_analysis

In [ ]:
query = """
SELECT
    category,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY category
ORDER BY total_purchases DESC;
"""

category_analysis = pd.read_sql_query(query, conn)

category_analysis

In [ ]:
query = """
SELECT
    item_purchased,
    COUNT(*) AS total_purchases
FROM customer
GROUP BY item_purchased
ORDER BY total_purchases DESC
LIMIT 10;
"""

top_items = pd.read_sql_query(query, conn)

top_items

In [ ]:
query = """
SELECT
    location,
    SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY location
ORDER BY total_revenue DESC
LIMIT 10;
"""

location_revenue = pd.read_sql_query(query, conn)

location_revenue

In [ ]:
query = """
SELECT
    subscription_status,
    COUNT(*) AS total_purchases,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase
FROM customer
GROUP BY subscription_status
ORDER BY total_purchases DESC;
"""

subscription_analysis = pd.read_sql_query(query, conn)

subscription_analysis

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(
    category_analysis["category"],
    category_analysis["total_purchases"]
)

plt.xlabel("Category")
plt.ylabel("Total Purchases")
plt.title("Total Purchases by Category")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    category_analysis["category"],
    category_analysis["total_revenue"]
)

plt.xlabel("Category")
plt.ylabel("Total Revenue")
plt.title("Total Revenue by Category")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    top_items["item_purchased"],
    top_items["total_purchases"]
)

plt.xlabel("Item Purchased")
plt.ylabel("Total Purchases")
plt.title("Top 10 Most Purchased Items")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    location_revenue["location"],
    location_revenue["total_revenue"]
)

plt.xlabel("Total Revenue")
plt.ylabel("Location")
plt.title("Top 10 Locations by Revenue")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    age_analysis["calculated_age_group"],
    age_analysis["total_purchases"]
)

plt.xlabel("Age Group")
plt.ylabel("Total Purchases")
plt.title("Total Purchases by Age Group")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    subscription_analysis["subscription_status"],
    subscription_analysis["total_purchases"]
)

plt.xlabel("Subscription Status")
plt.ylabel("Total Purchases")
plt.title("Total Purchases by Subscription Status")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    payment_analysis["payment_method"],
    payment_analysis["total_purchases"]
)

plt.xlabel("Payment Method")
plt.ylabel("Total Purchases")
plt.title("Purchases by Payment Method")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    discount_analysis["discount_applied"],
    discount_analysis["total_purchases"]
)

plt.xlabel("Discount Applied")
plt.ylabel("Total Purchases")
plt.title("Purchases With and Without Discount")

plt.tight_layout()
plt.show()

# **KPI SUMMARY SECTION**

In [ ]:
query = """
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT customer_id) AS total_customers,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS average_purchase_amount,
    ROUND(AVG(review_rating), 2) AS average_review_rating
FROM customer;
"""

kpi_summary = pd.read_sql_query(query, conn)

kpi_summary

In [ ]:
print("========== CUSTOMER SHOPPING BEHAVIOR KPI SUMMARY ==========")
print()

print("Total Transactions      :", kpi_summary.loc[0, "total_transactions"])
print("Total Customers         :", kpi_summary.loc[0, "total_customers"])
print("Total Revenue           :", kpi_summary.loc[0, "total_revenue"])
print("Average Purchase Amount :", kpi_summary.loc[0, "average_purchase_amount"])
print("Average Review Rating   :", kpi_summary.loc[0, "average_review_rating"])

In [ ]:
highest_revenue_category = category_analysis.loc[
    category_analysis["total_revenue"].idxmax()
]

highest_average_category = category_analysis.loc[
    category_analysis["average_purchase"].idxmax()
]

print("Category with Highest Revenue:")
print(highest_revenue_category)

print("\nCategory with Highest Average Purchase:")
print(highest_average_category)

In [ ]:
highest_revenue_location = location_revenue.loc[
    location_revenue["total_revenue"].idxmax()
]

highest_purchase_location = location_analysis.loc[
    location_analysis["total_purchases"].idxmax()
]

print("Location with Highest Revenue:")
print(highest_revenue_location)

print("\nLocation with Highest Number of Purchases:")
print(highest_purchase_location)

In [ ]:
highest_subscription_purchases = subscription_analysis.loc[
    subscription_analysis["total_purchases"].idxmax()
]

highest_subscription_average = subscription_analysis.loc[
    subscription_analysis["average_purchase"].idxmax()
]

print("Subscription Status with Highest Purchases:")
print(highest_subscription_purchases)

print("\nSubscription Status with Highest Average Purchase:")
print(highest_subscription_average)

In [ ]:
highest_discount_average = discount_analysis.loc[
    discount_analysis["average_purchase"].idxmax()
]

highest_discount_revenue = discount_analysis.loc[
    discount_analysis["total_revenue"].idxmax()
]

print("Discount Group with Highest Average Purchase:")
print(highest_discount_average)

print("\nDiscount Group with Highest Revenue:")
print(highest_discount_revenue)

In [ ]:
highest_rated_category = rating_category.loc[
    rating_category["average_rating"].idxmax()
]

overall_rating = rating_summary.loc[0, "average_rating"]

print("Overall Average Rating:")
print(overall_rating)

print("\nCategory with Highest Average Rating:")
print(highest_rated_category)

In [ ]:
highest_payment_purchases = payment_analysis.loc[
    payment_analysis["total_purchases"].idxmax()
]

highest_payment_average = payment_analysis.loc[
    payment_analysis["average_purchase"].idxmax()
]

print("Payment Method with Highest Purchases:")
print(highest_payment_purchases)

print("\nPayment Method with Highest Average Purchase:")
print(highest_payment_average)

In [ ]:
highest_shipping_purchases = shipping_analysis.loc[
    shipping_analysis["total_purchases"].idxmax()
]

highest_shipping_average = shipping_analysis.loc[
    shipping_analysis["average_purchase"].idxmax()
]

print("Shipping Method with Highest Purchases:")
print(highest_shipping_purchases)

print("\nShipping Method with Highest Average Purchase:")
print(highest_shipping_average)

In [ ]:
print("========== KEY BUSINESS INSIGHTS ==========\n")

print("1. Category Performance")
print(
    f"{highest_revenue_category['category']} generated the highest total revenue "
    f"of {highest_revenue_category['total_revenue']}."
)

print("\n2. Location Performance")
print(
    f"{highest_revenue_location['location']} generated the highest revenue "
    f"among the analyzed locations, with {highest_revenue_location['total_revenue']}."
)

print("\n3. Subscription Analysis")
print(
    f"Customers with subscription status '{highest_subscription_purchases['subscription_status']}' "
    f"accounted for the highest number of purchases: "
    f"{highest_subscription_purchases['total_purchases']}."
)

print("\n4. Discount Analysis")
print(
    f"The '{highest_discount_average['discount_applied']}' discount group "
    f"had the highest average purchase amount of "
    f"{highest_discount_average['average_purchase']}."
)

print("\n5. Customer Ratings")
print(
    f"The overall average review rating was {overall_rating}, while "
    f"{highest_rated_category['category']} had the highest category-level "
    f"average rating of {highest_rated_category['average_rating']}."
)

print("\n6. Payment Method")
print(
    f"{highest_payment_purchases['payment_method']} had the highest number "
    f"of purchases: {highest_payment_purchases['total_purchases']}."
)

print("\n7. Shipping Method")
print(
    f"{highest_shipping_purchases['shipping_type']} had the highest number "
    f"of purchases: {highest_shipping_purchases['total_purchases']}."
)

In [ ]:
final_summary = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Total Customers",
        "Total Revenue",
        "Average Purchase Amount",
        "Average Review Rating"
    ],
    "Value": [
        kpi_summary.loc[0, "total_transactions"],
        kpi_summary.loc[0, "total_customers"],
        kpi_summary.loc[0, "total_revenue"],
        kpi_summary.loc[0, "average_purchase_amount"],
        kpi_summary.loc[0, "average_review_rating"]
    ]
})

final_summary

In [ ]:
business_insights = pd.DataFrame({
    "Insight": [
        "Highest Revenue Category",
        "Highest Revenue Location",
        "Most Purchases by Subscription Status",
        "Highest Average Purchase - Discount Group",
        "Highest Rated Category",
        "Most Used Payment Method",
        "Most Used Shipping Method"
    ],
    "Result": [
        highest_revenue_category["category"],
        highest_revenue_location["location"],
        highest_subscription_purchases["subscription_status"],
        highest_discount_average["discount_applied"],
        highest_rated_category["category"],
        highest_payment_purchases["payment_method"],
        highest_shipping_purchases["shipping_type"]
    ]
})

business_insights

In [ ]:
final_business_insights = pd.DataFrame({
    "Business Area": [
        "Category",
        "Location",
        "Subscription",
        "Discount",
        "Customer Rating",
        "Payment",
        "Shipping"
    ],
    "Key Finding": [
        f"{highest_revenue_category['category']} generated the highest revenue",
        f"{highest_revenue_location['location']} generated the highest revenue",
        f"'{highest_subscription_purchases['subscription_status']}' had the most purchases",
        f"'{highest_discount_average['discount_applied']}' had the highest average purchase",
        f"{highest_rated_category['category']} had the highest average rating",
        f"{highest_payment_purchases['payment_method']} was the most used payment method",
        f"{highest_shipping_purchases['shipping_type']} was the most used shipping method"
    ]
})

final_business_insights

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

kpi_summary.to_csv("outputs/kpi_summary.csv", index=False)
category_analysis.to_csv("outputs/category_analysis.csv", index=False)
item_revenue.to_csv("outputs/item_revenue.csv", index=False)
location_revenue.to_csv("outputs/location_revenue.csv", index=False)
age_analysis.to_csv("outputs/age_analysis.csv", index=False)
subscription_analysis.to_csv("outputs/subscription_analysis.csv", index=False)
discount_analysis.to_csv("outputs/discount_analysis.csv", index=False)
payment_analysis.to_csv("outputs/payment_analysis.csv", index=False)
shipping_analysis.to_csv("outputs/shipping_analysis.csv", index=False)
rating_category.to_csv("outputs/rating_category.csv", index=False)
frequency_revenue_analysis.to_csv("outputs/frequency_revenue_analysis.csv", index=False)
final_business_insights.to_csv("outputs/final_business_insights.csv", index=False)

print("All analysis results have been saved successfully.")
print("\nFiles created:", len(os.listdir("outputs")))
print("\nSaved files:")
for file in os.listdir("outputs"):
    print("-", file)

In [ ]:
import matplotlib.pyplot as plt

metrics = [
    "Total Transactions",
    "Total Customers",
    "Total Revenue",
    "Average Purchase",
    "Average Rating"
]

values = [
    kpi_summary.loc[0, "total_transactions"],
    kpi_summary.loc[0, "total_customers"],
    kpi_summary.loc[0, "total_revenue"],
    kpi_summary.loc[0, "average_purchase_amount"],
    kpi_summary.loc[0, "average_review_rating"]
]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, metric, value in zip(axes, metrics, values):
    ax.text(
        0.5, 0.5,
        f"{value:,.2f}" if isinstance(value, float) else f"{value:,}",
        ha="center",
        va="center",
        fontsize=20
    )
    ax.set_title(metric, fontsize=12)
    ax.axis("off")

plt.suptitle(
    "Customer Shopping Behavior — KPI Dashboard",
    fontsize=18
)

plt.tight_layout()
plt.show()

## Conclusion

The Customer Shopping Behavior Analysis project provides a structured view of purchasing patterns using SQL and Python-based data analysis.

The analysis examined product categories, individual items, customer locations, age groups, subscription status, discounts, payment methods, shipping preferences, customer ratings, and purchase frequency.

Key findings were identified through SQL aggregation, Pandas analysis, and data visualizations. The results can help understand customer purchasing patterns, revenue distribution, product demand, and customer preferences.

Overall, this project demonstrates practical skills in:

- Data Cleaning and Exploration
- SQL Data Analysis
- Python and Pandas
- SQLite Database Querying
- Data Visualization with Matplotlib
- KPI Development
- Business Insight Generation

In [ ]:
required_variables = [
    "df",
    "kpi_summary",
    "category_analysis",
    "item_revenue",
    "location_revenue",
    "age_analysis",
    "subscription_analysis",
    "discount_analysis",
    "payment_analysis",
    "shipping_analysis",
    "rating_category",
    "frequency_revenue_analysis",
    "final_summary",
    "final_business_insights"
]

print("========== FINAL PROJECT CHECK ==========\n")

missing_variables = []

for variable in required_variables:
    if variable in globals():
        print("✓", variable)
    else:
        print("✗", variable, "MISSING")
        missing_variables.append(variable)

print("\n----------------------------------------")

if len(missing_variables) == 0:
    print("SUCCESS: All required project variables are available.")
else:
    print("Missing variables:", missing_variables)

In [ ]:
import os

project_folder = "customer-shopping-behavior-analysis"

os.makedirs(project_folder, exist_ok=True)
os.makedirs(f"{project_folder}/outputs", exist_ok=True)

print("Project folder created successfully:")
print(project_folder)
print(f"{project_folder}/outputs/")

In [ ]:
print("========== DATASET VERIFICATION ==========\n")

print("Rows       :", df.shape[0])
print("Columns    :", df.shape[1])
print("Column Names:")
print(list(df.columns))

print("\nDataset is available in the DataFrame.")

In [ ]:
dataset_path = "customer-shopping-behavior-analysis/customer_shopping_behavior.csv"

df.to_csv(dataset_path, index=False)

print("Dataset saved successfully!")
print("File:", dataset_path)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
import shutil
import os

source_folder = "outputs"
destination_folder = "customer-shopping-behavior-analysis/outputs"

for file in os.listdir(source_folder):
    source_path = os.path.join(source_folder, file)
    destination_path = os.path.join(destination_folder, file)

    if os.path.isfile(source_path):
        shutil.copy2(source_path, destination_path)

print("Analysis files copied successfully.\n")

print("Files inside project outputs folder:")
for file in sorted(os.listdir(destination_folder)):
    print("-", file)

print("\nTotal files:", len(os.listdir(destination_folder)))

In [ ]:
import os

print("========== OUTPUT FILES ==========\n")

files = os.listdir("outputs")

for i, file in enumerate(sorted(files), start=1):
    print(f"{i}. {file}")

print("\nTotal files:", len(files))

readme_content = """
# Customer Shopping Behavior Analysis

## Project Overview

This project analyzes customer shopping behavior using Python, Pandas, SQLite, SQL, and Matplotlib. The analysis focuses on purchasing patterns, product performance, customer demographics, revenue distribution, payment methods, shipping preferences, subscriptions, discounts, and customer ratings.

## Objectives

- Analyze customer purchasing behavior
- Identify high-performing product categories and products
- Analyze revenue across different locations
- Understand purchasing patterns across age groups
- Compare subscription and non-subscription customers
- Analyze payment and shipping preferences
- Examine purchases with and without discounts
- Analyze customer review ratings
- Generate business insights using SQL and Python

## Technologies Used

- Python
- Pandas
- SQLite
- SQL
- Matplotlib
- Google Colab

## Dataset

The project uses a customer shopping behavior dataset containing customer demographics, purchase information, product details, transaction attributes, ratings, subscription status, and purchasing frequency.

The dataset contains **3,900 customer purchase records**.

## Analysis Performed

### 1. Customer & Transaction Analysis
- Total transactions
- Total customers
- Average purchase amount
- Total revenue

### 2. Product Analysis
- Purchases by category
- Revenue by category
- Most purchased items
- Revenue by individual item

### 3. Customer Demographics
- Age-group analysis
- Location-based analysis
- Previous purchase behavior
- Purchase frequency

### 4. Customer Preferences
- Payment method analysis
- Shipping method analysis
- Subscription status analysis
- Discount analysis

### 5. Customer Satisfaction
- Overall review rating
- Average rating by product category

## Key Business Insights

The project identifies differences in purchasing volume, revenue contribution, customer ratings, payment preferences, shipping preferences, subscription activity, and discount usage.

Detailed results are available in the `outputs` directory.

## Project Structure

```text
customer-shopping-behavior-analysis/
│
├── Customer_Shopping_Behavior_Analysis.ipynb
├── customer_shopping_behavior.csv
│
└── outputs/
    ├── age_analysis.csv
    ├── category_analysis.csv
    ├── discount_analysis.csv
    ├── final_business_insights.csv
    ├── frequency_revenue_analysis.csv
    ├── item_revenue.csv
    ├── kpi_summary.csv
    ├── location_revenue.csv
    ├── payment_analysis.csv
    ├── rating_category.csv
    ├── shipping_analysis.csv
    └── subscription_analysis.csv

In [ ]:
import os

project_folder = "customer-shopping-behavior-analysis"

print("========== GITHUB PROJECT STRUCTURE ==========\n")

for root, dirs, files in os.walk(project_folder):
    level = root.replace(project_folder, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in sorted(files):
        print(f"{indent}    └── {file}")

In [ ]:
import shutil
import os

project_folder = "customer-shopping-behavior-analysis"

zip_file = shutil.make_archive(
    project_folder,
    "zip",
    project_folder
)

print("GitHub project package created successfully!")
print("File:", zip_file)
print("Size:", round(os.path.getsize(zip_file) / 1024, 2), "KB")

In [ ]:
from google.colab import files

files.download("customer-shopping-behavior-analysis.zip")